
##Lesson 1: tracing
Why it matters
Every judge, every latency number and every token count in this project is read from a trace.


In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0" openai datasets
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow.entities import Document
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()  # auto-traces every LLM call: tokens, latency
client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-gpt-oss-120b"  # replace with a chat model from your Serving page

DOCS = [
    Document(page_content="Delta Lake is an open table format that adds ACID transactions to data lakes.", metadata={"doc_uri": "d1"}),
    Document(page_content="MLflow Tracing records each step of a GenAI request as spans.", metadata={"doc_uri": "d2"}),
    Document(page_content="Hyderabad is the capital of Telangana.", metadata={"doc_uri": "d3"}),
]

@mlflow.trace(span_type="RETRIEVER")
def retrieve(question: str, k: int = 2):
    words = set(question.lower().split())
    return sorted(DOCS, key=lambda d: -len(words & set(d.page_content.lower().split())))[:k]

@mlflow.trace
def answer(question: str) -> str:
    docs = retrieve(question)
    context = "\n".join(d.page_content for d in docs)
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "Answer only from the context. If it isn't there, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    return resp.choices[0].message.content

answer("What does MLflow Tracing record?")

In [0]:
from datasets import load_dataset
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation[:5]")
print(ds)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
print(sorted(e.name for e in w.serving_endpoints.list() if e.name.startswith("databricks-")))

In [0]:
%sql
SELECT ai_classify(
  'Question: What is the capital of Telangana?
   Context: Hyderabad is the capital of Telangana.
   Answer: Hyderabad.',
  ARRAY('grounded', 'not_grounded')
) AS verdict

In [0]:
r = ds[0]
print(r)
print("Q:", r["question"]); print("A:", r["answer"], "|", r["type"], r["level"])
print("Supporting:", list(zip(r["supporting_facts"]["title"], r["supporting_facts"]["sent_id"])))
for t, sents in zip(r["context"]["title"], r["context"]["sentences"]):
    print(f"\n[{t}] {len(sents)} sentences"); print(" ", sents[0][:120])

## Optimized Schema & Data Model for HotPotQA

The raw dataset packs **parallel arrays** (`context.titles` ↔ `context.sentences`, `supporting_facts.title` ↔ `supporting_facts.sent_id`) into a single row. This is hard to query, hard to join, and wastes shuffle bytes.

### Recommended: 3-table normalized model

| Table | Grain | Purpose |
|---|---|---|
| **eval_questions** | 1 row per question | Fact table — question text, expected answer, type, difficulty |
| **eval_passages** | 1 row per (question, passage) | All 10 distractor+gold passages, with `is_supporting` flag for retrieval evaluation |
| **eval_supporting_sentences** | 1 row per (question, supporting sentence) | Sentence-level ground truth for fine-grained judges |

**Why this layout?**
* **Retrieval eval** → JOIN `eval_questions` ↔ `eval_passages` WHERE `is_supporting = TRUE`, compute precision/recall on the title set.
* **Judge inputs** → each passage row already has the concatenated `content` string ready for the LLM prompt — no runtime array explosion.
* **Aggregation** → `question_type` and `difficulty` live on the fact table, so roll-ups are a simple GROUP BY with no self-join.
* **Delta benefits** — Z-ORDER on `question_id` across all three tables keeps co-located data in the same files; `question_type` is a natural partition candidate at scale (only 2 values: bridge / comparison).

In [0]:
from datasets import load_dataset
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType, ArrayType
)

# ── Load full validation split ──────────────────────────────────────
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")

# ── 1. eval_questions  (one row per question) ──────────────────────
questions_rows = [
    (r["id"], r["question"], r["answer"], r["type"], r["level"])
    for r in ds
]
questions_schema = StructType([
    StructField("question_id", StringType()),
    StructField("question", StringType()),
    StructField("expected_answer", StringType()),
    StructField("question_type", StringType()),
    StructField("difficulty", StringType()),
])
df_questions = spark.createDataFrame(questions_rows, schema=questions_schema)

# ── 2. eval_passages  (one row per question × passage) ─────────────
passage_rows = []
for r in ds:
    gold_titles = set(r["supporting_facts"]["title"])
    for idx, (title, sents) in enumerate(
        zip(r["context"]["title"], r["context"]["sentences"])
    ):
        passage_rows.append((
            r["id"],
            idx,
            title,
            " ".join(s.strip() for s in sents),
            title in gold_titles,
        ))
passages_schema = StructType([
    StructField("question_id", StringType()),
    StructField("passage_idx", IntegerType()),
    StructField("title", StringType()),
    StructField("content", StringType()),
    StructField("is_supporting", BooleanType()),
])
df_passages = spark.createDataFrame(passage_rows, schema=passages_schema)

# ── 3. eval_supporting_sentences  (sentence-level ground truth) ────
sf_rows = []
for r in ds:
    titles = r["context"]["title"]
    sents_map = dict(zip(titles, r["context"]["sentences"]))
    for sf_title, sf_sid in zip(
        r["supporting_facts"]["title"], r["supporting_facts"]["sent_id"]
    ):
        sent_text = ""
        if sf_title in sents_map and sf_sid < len(sents_map[sf_title]):
            sent_text = sents_map[sf_title][sf_sid].strip()
        sf_rows.append((r["id"], sf_title, sf_sid, sent_text))

sf_schema = StructType([
    StructField("question_id", StringType()),
    StructField("title", StringType()),
    StructField("sent_id", IntegerType()),
    StructField("sentence_text", StringType()),
])
df_supporting = spark.createDataFrame(sf_rows, schema=sf_schema)

# ── Quick sanity check ──────────────────────────────────────────────
print(f"eval_questions          : {df_questions.count():,} rows")
print(f"eval_passages           : {df_passages.count():,} rows  (≈10 per question)")
print(f"eval_supporting_sentences: {df_supporting.count():,} rows (≈2-5 per question)")

print("\n── eval_questions ──")
df_questions.printSchema()
df_questions.show(3, truncate=60)

print("── eval_passages ──")
df_passages.printSchema()
df_passages.show(3, truncate=80)

print("── eval_supporting_sentences ──")
df_supporting.printSchema()
df_supporting.show(3, truncate=80)

In [0]:
# ── Example query: retrieval-style join for evaluation ──────────────
# Register as temp views so we can use SQL-style logic
df_questions.createOrReplaceTempView("eval_questions")
df_passages.createOrReplaceTempView("eval_passages")
df_supporting.createOrReplaceTempView("eval_supporting_sentences")

# How many gold vs distractor passages per question type?
result = spark.sql("""
    SELECT q.question_type,
           p.is_supporting,
           COUNT(*)             AS passage_count,
           ROUND(AVG(LENGTH(p.content)))  AS avg_passage_chars
      FROM eval_questions q
      JOIN eval_passages  p USING (question_id)
  GROUP BY q.question_type, p.is_supporting
  ORDER BY q.question_type, p.is_supporting DESC
""")
display(result)

In [0]:
%pip install rank_bm25
dbutils.library.restartPython()

In [0]:
import numpy as np, re
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from databricks.sdk import WorkspaceClient

r = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation[:1]")[0]
q = r["question"]
titles = r["context"]["title"]
paras = [" ".join(s) for s in r["context"]["sentences"]]
print("Q:", q, "| needed:", set(r["supporting_facts"]["title"]))

# ---- 1. SPARSE: what BM25 actually sees ----
tok = lambda t: re.findall(r"[a-z0-9]+", t.lower())
print("\nquery tokens:", tok(q))
bm25 = BM25Okapi([tok(p) for p in paras])
for w in tok(q):
    print(f"  IDF({w:12}) = {bm25.idf.get(w, 0):.2f}")       # rarity of each word
bm = bm25.get_scores(tok(q))

# ---- 2. DENSE: what an embedding actually is ----
client = WorkspaceClient().serving_endpoints.get_open_ai_client()
EMB = "databricks-gte-large-en"   # replace with the embedding model from your list
vecs = np.array([d.embedding for d in client.embeddings.create(model=EMB, input=[q] + paras).data])
print("\nvector shape:", vecs.shape, "| first 5 numbers of the question:", vecs[0][:5].round(3))
unit = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
cos = unit[1:] @ unit[0]                                       # cosine = dot product of unit vectors

# ---- 3. SIDE BY SIDE ----
print(f"\n{'paragraph':32} {'BM25':>6} {'BM25 rank':>9} {'cosine':>7} {'dense rank':>10}")
br, dr = (-bm).argsort().argsort() + 1, (-cos).argsort().argsort() + 1
for i, t in enumerate(titles):
    mark = " ✓" if t in r["supporting_facts"]["title"] else ""
    print(f"{(t + mark)[:32]:32} {bm[i]:6.2f} {br[i]:9} {cos[i]:7.3f} {dr[i]:10}")

In [0]:
'Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."'